In [7]:
!pip install pandas awswrangler geopandas matplotlib


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
import os
os.environ['AWS_DEFAULT_REGION'] = 'us-east-2'
# No SageMaker a autenticação é automática via IAM Role

In [10]:
# =====================================================================
# Mapa Coroplético do Brasil — Risco de Sobrecarga Hospitalar por UF
# =====================================================================
#
# Dependências (rodar em uma célula anterior):
#   !pip install pandas awswrangler geopandas matplotlib
#
# Pré-requisito: Credenciais AWS configuradas (via SageMaker, variáveis
# de ambiente AWS_ACCESS_KEY_ID / AWS_SECRET_ACCESS_KEY, ou perfil IAM).
#
# O mapa será exibido diretamente na saída da célula.
# =====================================================================

import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patheffects as pe
from matplotlib.patches import Patch
import awswrangler as wr
import re

# =====================================================================
# 1. LEITURA DOS DADOS REAIS DA CAMADA GOLD (S3)
# =====================================================================
S3_GOLD_PATH = "s3://dressasys-sagemakers-fiap/analise_covid19/inteligencia_hospitalar_gold/"

print("1. Lendo dados reais da Camada Gold (Gold_Alerta_Risco_Hospitalar)...")
df_gold = wr.s3.read_parquet(
    path=f"{S3_GOLD_PATH}Gold_Alerta_Risco_Hospitalar/",
    dataset=True
)

print(f"   Registros carregados: {len(df_gold):,}")
print(f"   Colunas: {df_gold.columns.tolist()}")

# =====================================================================
# 2. EXTRAIR UF E AGREGAR PACIENTES GRAVES POR ESTADO
# =====================================================================
print("2. Agregando pacientes graves por UF...")

# Extrair sigla UF do campo polo_epidemiologico (ex: 'Capital (SP)' -> 'SP')
df_gold['uf'] = df_gold['polo_epidemiologico'].apply(
    lambda x: re.search(r'\((\w{2})\)', x).group(1)
)

# Agregar total de pacientes graves estimados por UF
df_uf = df_gold.groupby('uf').agg(
    total_pacientes_graves=('total_pacientes_estimados', 'sum')
).reset_index()

# =====================================================================
# 3. LEITOS UTI POR UF — Estimativa baseada em dados CNES/DataSUS
# =====================================================================
# Leitos UTI por 100 mil habitantes (estimativa — fonte: CNES/DataSUS)
LEITOS_UTI_POR_100K = {
    'AC': 8.2,  'AL': 7.5,  'AM': 7.0,  'AP': 5.8,
    'BA': 9.1,  'CE': 10.2, 'DF': 22.5, 'ES': 14.3,
    'GO': 13.1, 'MA': 5.5,  'MG': 12.8, 'MS': 14.0,
    'MT': 13.5, 'PA': 6.2,  'PB': 9.8,  'PE': 11.5,
    'PI': 8.0,  'PR': 15.2, 'RJ': 18.0, 'RN': 10.5,
    'RO': 9.0,  'RR': 6.5,  'RS': 16.8, 'SC': 17.0,
    'SE': 8.5,  'SP': 19.5, 'TO': 7.8
}

# População por UF (IBGE 2022)
POPULACAO_UF = {
    'AC': 906876,   'AL': 3365351,  'AM': 4207714,  'AP': 877613,
    'BA': 14930634, 'CE': 9240580,  'DF': 3094325,  'ES': 4108508,
    'GO': 7206589,  'MA': 7153262,  'MG': 21411923, 'MS': 2839188,
    'MT': 3567234,  'PA': 8777124,  'PB': 4059905,  'PE': 9674793,
    'PI': 3289290,  'PR': 11597484, 'RJ': 17463349, 'RN': 3560903,
    'RO': 1815278,  'RR': 652713,   'RS': 11466630, 'SC': 7338473,
    'SE': 2338474,  'SP': 46649132, 'TO': 1607363
}

# Calcular leitos UTI totais estimados e Índice de Pressão Hospitalar
df_uf['populacao'] = df_uf['uf'].map(POPULACAO_UF)
df_uf['leitos_uti_por_100k'] = df_uf['uf'].map(LEITOS_UTI_POR_100K)
df_uf['leitos_uti_estimados'] = (df_uf['populacao'] / 100_000) * df_uf['leitos_uti_por_100k']
df_uf['indice_pressao_hospitalar'] = (
    (df_uf['total_pacientes_graves'] / df_uf['leitos_uti_estimados']) * 100
).round(1)

# =====================================================================
# 4. RANKING — Top 10 UFs com maior Índice de Pressão Hospitalar
# =====================================================================
print("\nTop 10 UFs com maior Índice de Pressão Hospitalar (DADOS REAIS S3):")
print("-" * 70)
ranking = df_uf.sort_values('indice_pressao_hospitalar', ascending=False)
for i, (_, row) in enumerate(ranking.head(10).iterrows(), 1):
    print(
        f"  {i:2d}. {row['uf']}  |  "
        f"Graves: {row['total_pacientes_graves']:>12,.0f}  |  "
        f"Leitos UTI: {row['leitos_uti_estimados']:>7,.0f}  |  "
        f"Índice: {row['indice_pressao_hospitalar']:>10,.1f}"
    )

# =====================================================================
# 5. CARREGANDO O SHAPEFILE DO BRASIL
# =====================================================================
print("\n3. Carregando geometria dos estados brasileiros...")
url_geojson = (
    'https://raw.githubusercontent.com/codeforamerica/'
    'click_that_hood/master/public/data/brazil-states.geojson'
)
gdf_brasil = gpd.read_file(url_geojson)
gdf_brasil = gdf_brasil.merge(df_uf, left_on='sigla', right_on='uf', how='left')

# =====================================================================
# 6. MAPA COROPLÉTICO — ÍNDICE DE PRESSÃO HOSPITALAR
# =====================================================================
print("4. Gerando mapa coroplético...")

fig, ax = plt.subplots(1, 1, figsize=(14, 12))

vmin = df_uf['indice_pressao_hospitalar'].min()
vmax = df_uf['indice_pressao_hospitalar'].max()

# Escala de cores: Verde (baixo risco) -> Amarelo -> Vermelho (alto risco)
cmap = mcolors.LinearSegmentedColormap.from_list(
    'risco_hospitalar',
    ['#2ecc71', '#f1c40f', '#e67e22', '#e74c3c', '#8b0000'],
    N=256,
)

gdf_brasil.plot(
    column='indice_pressao_hospitalar',
    cmap=cmap,
    linewidth=0.8,
    edgecolor='#2c3e50',
    ax=ax,
    legend=False,
    vmin=vmin,
    vmax=vmax,
)

# Rótulos: sigla + índice no centróide de cada estado
for _, row in gdf_brasil.iterrows():
    if pd.notna(row.get('indice_pressao_hospitalar')):
        centroid = row['geometry'].centroid
        label = f"{row['sigla']}\n{row['indice_pressao_hospitalar']:.0f}"
        ax.annotate(
            label,
            xy=(centroid.x, centroid.y),
            ha='center', va='center',
            fontsize=7, fontweight='bold', color='white',
            path_effects=[pe.withStroke(linewidth=2, foreground='black')],
        )

# Barra de cores (colorbar)
sm = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(vmin=vmin, vmax=vmax))
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax, fraction=0.03, pad=0.04)
cbar.set_label(
    'Índice de Pressão Hospitalar\n(Pacientes Graves / Leitos UTI × 100)',
    fontsize=11, fontweight='bold',
)

# Legenda interpretativa (faixas baseadas nos dados reais)
legend_elements = [
    Patch(facecolor='#2ecc71', edgecolor='black', label='Menor Pressão (< 20.000)'),
    Patch(facecolor='#f1c40f', edgecolor='black', label='Pressão Moderada (20.000–35.000)'),
    Patch(facecolor='#e67e22', edgecolor='black', label='Pressão Alta (35.000–50.000)'),
    Patch(facecolor='#e74c3c', edgecolor='black', label='Pressão Crítica (50.000–70.000)'),
    Patch(facecolor='#8b0000', edgecolor='black', label='Colapso (> 70.000)'),
]
ax.legend(
    handles=legend_elements, loc='lower left', fontsize=9,
    title='Classificação de Risco', title_fontsize=10,
    framealpha=0.9, fancybox=True,
)

# Títulos e ajustes visuais
ax.set_title(
    'Mapa Coroplético: Risco de Sobrecarga Hospitalar por UF\n'
    '(Índice = Pacientes Graves Estimados / Leitos UTI × 100)\n'
    'Fonte: Camada Gold — S3 dressasys-sagemakers-fiap',
    fontsize=14, fontweight='bold', pad=20,
)
ax.set_xlabel('Longitude', fontsize=10)
ax.set_ylabel('Latitude', fontsize=10)
ax.set_facecolor('#f0f0f0')

fig.text(
    0.5, 0.02,
    'Fonte: Dados reais da PNAD COVID-19 (IBGE) — Camada Gold (Gold_Alerta_Risco_Hospitalar). '
    'Leitos UTI estimados com base em dados CNES/DataSUS.',
    ha='center', fontsize=8, style='italic', color='gray',
)

plt.tight_layout()
plt.show()

print("\nMapa gerado com sucesso a partir dos dados REAIS do S3!")


1. Lendo dados reais da Camada Gold (Gold_Alerta_Risco_Hospitalar)...


NoCredentialsError: Unable to locate credentials